# Online Retail Analytics - SQL Case Study

In [2]:
import pandas as pd
from sqlalchemy import create_engine

In [3]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

password = "yourpassword"

connection_url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="Online Retail"
)

engine = create_engine(connection_url)

print("PostgreSQL connection successful")

PostgreSQL connection successful


In [16]:
query = """
SELECT COUNT(*) AS total_customers
FROM customers;
"""

df = pd.read_sql(query, engine)

df

,total_customers
0,1000


### Q1: Retrieve top 5 highest value completed orders (order_id, customer_id, order_amount, order_date)


In [17]:
q1 = """
SELECT
    o.order_id,
    o.customer_id,
    SUM(oi.quantity * oi.unit_price) AS order_amount,
    o.order_date
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
WHERE o.status = 'Completed'
GROUP BY
    o.order_id,
    o.customer_id,
    o.order_date
ORDER BY order_amount DESC
LIMIT 5;
"""

q1_result = pd.read_sql(q1, engine)

q1_result

,order_id,customer_id,order_amount,order_date
0,300,301,4994.55,2025-11-18
1,795,796,4987.75,2025-10-04
2,45,46,4846.60,2025-11-03
3,180,181,4833.15,2025-12-18
4,15,16,4760.50,2025-12-03


### Insights
The top 5 completed orders range from 4,760.50 to 4,994.55 in order value. These orders represent the highest-value individual transactions in the dataset.

### Q2: List customers with more than 2 orders (customer_id, full_name, total_orders)


In [ ]:
q2 = """
    SELECT
        c.customer_id,
        c.full_name,
        COUNT(o.order_id) AS total_orders
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY
        c.customer_id,
        c.full_name
    HAVING COUNT(o.order_id) > 2
    ORDER BY total_orders DESC;
    """

    q2_result = pd.read_sql(q2, engine)

    q2_result

,customer_id,full_name,total_orders


### Insights
No customers placed more than 2 orders. The result is empty, indicating that the dataset does not contain customers meeting the repeat-order threshold used in this query.

### Q3: Get customers who never placed an order using LEFT JOIN (customer_id, full_name, signup_date)


In [4]:
q3 = """
SELECT
    c.customer_id,
    c.full_name,
    c.signup_date
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;
"""

q3_result = pd.read_sql(q3, engine)

q3_result

,customer_id,full_name,signup_date


### Insights
No customers were found without an order. This means every customer in the dataset is associated with at least one order.

### Q4: Show most expensive product purchased by each customer (customer_id, product_name, price)


In [ ]:
q4 = """
SELECT
    o.customer_id,
    p.product_name,
    p.price
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
JOIN products p
    ON oi.product_id = p.product_id
WHERE p.price = (   
    SELECT MAX(p2.price)    
    FROM orders o2
    JOIN order_items oi2
        ON o2.order_id = oi2.order_id
    JOIN products p2
        ON oi2.product_id = p2.product_id
    WHERE o2.customer_id = o.customer_id
);
"""

q4_result = pd.read_sql(q4, engine)

q4_result

,customer_id,product_name,price
0,3,Product 2,224.45
1,4,Product 3,315.16
2,5,Product 4,944.79
3,6,Product 5,544.91
4,7,Product 6,837.44
...,...,...,...
994,998,Product 997,629.16
995,999,Product 998,909.16
996,1000,Product 999,318.47
997,1,Product 1000,952.00


### Insights
The query returns the highest-priced purchased product for each customer. The results contain 999 customers, showing that almost all customers have a matching purchase record.

### Q5: Find order with largest number of items (order_id, total_items)


In [14]:
q5 = """
WITH order_totals AS (
    SELECT
        order_id,
        SUM(quantity) AS total_items
    FROM order_items
    GROUP BY order_id
)
SELECT
    order_id,
    total_items
FROM order_totals
WHERE total_items = (
    SELECT MAX(total_items)
    FROM order_totals
)
ORDER BY order_id;
"""

q5_result = pd.read_sql(q5, engine)

q5_result

,order_id,total_items
0,5,5
1,10,5
2,15,5
3,20,5
4,25,5
...,...,...
195,980,5
196,985,5
197,990,5
198,995,5


### Insights
The maximum number of items in an order is 5. The result shows multiple orders with this quantity, so 5 items is the largest basket size in the dataset.

### Q6: List customers who ordered within 30 days of signup (customer_id, full_name, signup_date, first_order_date, days_difference)


In [13]:
q6 = """
WITH first_orders AS (
    SELECT
        customer_id,
        MIN(order_date) AS first_order_date
    FROM orders
    GROUP BY customer_id
)
 
SELECT
    c.customer_id,
    c.full_name,
    c.signup_date,
    f.first_order_date,
    (f.first_order_date - c.signup_date) AS days_difference
FROM customers AS c
INNER JOIN first_orders AS f
    ON c.customer_id = f.customer_id
WHERE
    (f.first_order_date - c.signup_date) BETWEEN 0 AND 30
ORDER BY
    days_difference ASC,
    c.customer_id;
"""

q6_result = pd.read_sql(q6, engine)

q6_result

,customer_id,full_name,signup_date,first_order_date,days_difference
0,2,Customer 2,2025-12-16,2025-12-17,1
1,3,Customer 3,2025-12-15,2025-12-16,1
2,4,Customer 4,2025-12-14,2025-12-15,1
3,5,Customer 5,2025-12-13,2025-12-14,1
4,6,Customer 6,2025-12-12,2025-12-13,1
...,...,...,...,...,...
84,86,Customer 86,2025-09-23,2025-09-24,1
85,87,Customer 87,2025-09-22,2025-09-23,1
86,88,Customer 88,2025-09-21,2025-09-22,1
87,89,Customer 89,2025-09-20,2025-09-21,1


### Insights
89 customers placed their first order within 30 days of signup. In the displayed results, the first order occurred just 1 day after signup, showing quick initial purchasing activity for these customers.

### Q7: Count orders per status per country (country, status, order_count)


In [15]:
q7 = """
SELECT
    c.country,
    o.status,
    COUNT(o.order_id) AS order_count
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
GROUP BY c.country, o.status
ORDER BY c.country, o.status;
"""
q7_result = pd.read_sql(q7, engine)
q7_result

,country,status,order_count
0,Germany,Cancelled,83
1,Germany,Completed,83
2,Germany,Pending,84
3,India,Cancelled,83
4,India,Completed,84
5,India,Pending,83
6,UK,Cancelled,84
7,UK,Completed,83
8,UK,Pending,83
9,USA,Cancelled,84


### Insights
Order counts are very evenly distributed across Germany, India, the UK, and the USA. Each country has roughly 83–84 orders for each status, showing a balanced sample across countries and order statuses.

### Q8: Calculate Customer Lifetime Value (customer_id, full_name, total_revenue)


In [16]:
q8 = """
SELECT
    c.customer_id,
    c.full_name,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.full_name
ORDER BY total_revenue DESC;
"""
q8_result = pd.read_sql(q8, engine)
q8_result

,customer_id,full_name,total_revenue
0,301,Customer 301,4994.55
1,796,Customer 796,4987.75
2,591,Customer 591,4974.25
3,641,Customer 641,4963.85
4,431,Customer 431,4923.50
...,...,...,...
995,492,Customer 492,119.27
996,182,Customer 182,119.02
997,717,Customer 717,109.68
998,502,Customer 502,106.12


### Insights
Customer revenue varies considerably, with the highest customer revenue reaching 4,994.55. The results can be used to identify customers contributing the most revenue to the business.

### Q9: Identify repeat customers - more than 1 order per month (customer_id, full_name, order_month, monthly_order_count)


In [17]:
q9 = """
SELECT
    c.customer_id,
    c.full_name,
    DATE_TRUNC('month', o.order_date) AS order_month,
    COUNT(o.order_id) AS monthly_order_count
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.full_name, DATE_TRUNC('month', o.order_date)
HAVING COUNT(o.order_id) > 1
ORDER BY c.customer_id, order_month;
"""
q9_result = pd.read_sql(q9, engine)
q9_result

,customer_id,full_name,order_month,monthly_order_count


### Insights
No customers placed more than one order in the same month. Therefore, the dataset does not contain customers meeting the repeat-purchase condition used in this analysis.

### Q10: Monthly revenue trend (year, month, total_revenue)


In [18]:
q10 = """
SELECT
    EXTRACT(YEAR FROM o.order_date) AS year,
    EXTRACT(MONTH FROM o.order_date) AS month,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
FROM orders AS o
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
GROUP BY EXTRACT(YEAR FROM o.order_date), EXTRACT(MONTH FROM o.order_date)
ORDER BY year, month;
"""
q10_result = pd.read_sql(q10, engine)
q10_result

,year,month,total_revenue
0,2025.0,9.0,216252.11
1,2025.0,10.0,570734.04
2,2025.0,11.0,576401.97
3,2025.0,12.0,342316.63


### Insights
Revenue increased from 216,252.11 in September to 570,734.04 in October and 576,401.97 in November, before falling to 342,316.63 in December. November was the highest-revenue month in the dataset.

### Q11: Most returned products where status=Returned (product_id, product_name, return_count)


In [19]:
q11 = """
SELECT
    p.product_id,
    p.product_name,
    COUNT(*) AS return_count
FROM orders AS o
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
INNER JOIN products AS p
    ON oi.product_id = p.product_id
WHERE o.status = 'Returned'
GROUP BY p.product_id, p.product_name
ORDER BY return_count DESC, p.product_id;
"""
q11_result = pd.read_sql(q11, engine)
q11_result

,product_id,product_name,return_count


### Insights
No returned products were found using the Returned order-status condition. Therefore, the dataset contains no records matching the return condition used in this query.

### Q12: Top 3 revenue-generating categories (category, total_revenue)


In [20]:
q12 = """
SELECT
    p.category,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
FROM order_items AS oi
INNER JOIN products AS p
    ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC
LIMIT 3;
"""
q12_result = pd.read_sql(q12, engine)
q12_result

,category,total_revenue
0,Electronics,564726.80
1,Grocery,471850.60
2,Books,339531.66


### Insights
Electronics generated the highest revenue at 564,726.80, followed by Grocery at 471,850.60 and Books at 339,531.66. Electronics is therefore the strongest revenue-generating category among the three.

### Q13: Products frequently bought together - basket analysis (product_id_1, product_id_2, times_bought_together)


In [21]:
q13 = """
SELECT
    oi1.product_id AS product_id_1,
    oi2.product_id AS product_id_2,
    COUNT(DISTINCT oi1.order_id) AS times_bought_together
FROM order_items AS oi1
INNER JOIN order_items AS oi2
    ON oi1.order_id = oi2.order_id
    AND oi1.product_id < oi2.product_id
GROUP BY oi1.product_id, oi2.product_id
ORDER BY times_bought_together DESC, product_id_1, product_id_2;
"""
q13_result = pd.read_sql(q13, engine)
q13_result

,product_id_1,product_id_2,times_bought_together


### Insights
No product pairs were returned by the basket analysis. This means the current dataset does not contain product combinations that satisfy the query's criteria.

### Q14: Customer retention - ordered again within 90 days (customer_id, full_name, first_order_date, next_order_date, days_between_orders)


In [22]:
q14 = """
WITH customer_orders AS (
    SELECT
        o.customer_id,
        o.order_date,
        LEAD(o.order_date) OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date
        ) AS next_order_date
    FROM orders AS o
)
SELECT
    c.customer_id,
    c.full_name,
    co.order_date AS first_order_date,
    co.next_order_date,
    (co.next_order_date - co.order_date) AS days_between_orders
FROM customer_orders AS co
INNER JOIN customers AS c
    ON co.customer_id = c.customer_id
WHERE
    co.next_order_date IS NOT NULL
    AND (co.next_order_date - co.order_date) BETWEEN 1 AND 90
ORDER BY days_between_orders ASC, c.customer_id;
"""
q14_result = pd.read_sql(q14, engine)
q14_result

,customer_id,full_name,first_order_date,next_order_date,days_between_orders


### Insights
No customers were found placing another order within 90 days. This indicates that the dataset does not contain customers meeting the retention condition used in this query.

### Q15: Net Revenue After Discounts & Tax monthly country-wise (country, year, month, gross_revenue, total_discount, total_tax, net_revenue)


In [23]:
q15 = """
SELECT
    c.country,
    EXTRACT(YEAR FROM o.order_date) AS year,
    EXTRACT(MONTH FROM o.order_date) AS month,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS gross_revenue,
    ROUND(SUM(o.discount_amount), 2) AS total_discount,
    ROUND(SUM(o.tax_amount), 2) AS total_tax,
    ROUND(SUM(oi.quantity * oi.unit_price) - SUM(o.discount_amount) + SUM(o.tax_amount), 2) AS net_revenue
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
GROUP BY c.country, EXTRACT(YEAR FROM o.order_date), EXTRACT(MONTH FROM o.order_date)
ORDER BY c.country, year, month;
"""
q15_result = pd.read_sql(q15, engine)
q15_result

,country,year,month,gross_revenue,total_discount,total_tax,net_revenue
0,Germany,2025.0,9.0,48250.47,970.60,563.22,47843.09
1,Germany,2025.0,10.0,159559.39,4610.13,2122.44,157071.70
2,Germany,2025.0,11.0,153645.31,4568.56,1904.08,150980.83
3,Germany,2025.0,12.0,81693.63,2809.13,1419.61,80304.11
4,India,2025.0,9.0,61626.42,1514.23,735.07,60847.26
5,India,2025.0,10.0,132323.24,4500.96,2257.38,130079.66
6,India,2025.0,11.0,147122.45,3861.25,1920.52,145181.72
7,India,2025.0,12.0,89122.98,2577.75,1483.04,88028.27
8,UK,2025.0,9.0,54306.36,1452.03,815.17,53669.50
9,UK,2025.0,10.0,147754.33,4134.17,2059.59,145679.75


### Insights
Net revenue is lower than gross revenue after accounting for discounts and tax. October and November generally show the highest revenue across the countries, with Germany recording the highest monthly net revenue of 157,071.70 in October.

### Q16: Discount Sensitivity - customers who only order with discounts (customer_id, full_name, total_orders, discounted_orders)


In [24]:
q16 = """
SELECT
    c.customer_id,
    c.full_name,
    COUNT(o.order_id) AS total_orders,
    COUNT(CASE WHEN o.discount_amount > 0 THEN 1 END) AS discounted_orders
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.full_name
HAVING COUNT(o.order_id) = COUNT(CASE WHEN o.discount_amount > 0 THEN 1 END)
ORDER BY c.customer_id;
"""
q16_result = pd.read_sql(q16, engine)
q16_result

,customer_id,full_name,total_orders,discounted_orders
0,1,Customer 1,1,1
1,2,Customer 2,1,1
2,3,Customer 3,1,1
3,4,Customer 4,1,1
4,5,Customer 5,1,1
...,...,...,...,...
995,996,Customer 996,1,1
996,997,Customer 997,1,1
997,998,Customer 998,1,1
998,999,Customer 999,1,1


### Insights
All 1,000 customers have one order and that order received a discount. Therefore, every customer in this dataset meets the condition of having all their orders discounted.

### Q17: First vs Most Recent Purchase Product using ROW_NUMBER (customer_id, first_product_name, last_product_name)


In [25]:
q17 = """
WITH ranked_orders AS (
    SELECT
        o.customer_id,
        p.product_name,
        ROW_NUMBER() OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date, o.order_id
        ) AS first_rank,
        ROW_NUMBER() OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date DESC, o.order_id DESC
        ) AS last_rank
    FROM orders AS o
    INNER JOIN order_items AS oi
        ON o.order_id = oi.order_id
    INNER JOIN products AS p
        ON oi.product_id = p.product_id
)
SELECT
    customer_id,
    MAX(CASE WHEN first_rank = 1 THEN product_name END) AS first_product_name,
    MAX(CASE WHEN last_rank = 1 THEN product_name END) AS last_product_name
FROM ranked_orders
GROUP BY customer_id
ORDER BY customer_id;
"""
q17_result = pd.read_sql(q17, engine)
q17_result

,customer_id,first_product_name,last_product_name
0,1,Product 1000,Product 1000
1,2,Product 1,Product 1
2,3,Product 2,Product 2
3,4,Product 3,Product 3
4,5,Product 4,Product 4
...,...,...,...
994,996,Product 995,Product 995
995,997,Product 996,Product 996
996,998,Product 997,Product 997
997,999,Product 998,Product 998


### Insights
The first and most recent purchased products are the same for all 999 returned customers. This is consistent with the dataset having customers with only one recorded order.

### Q18: Revenue Contribution Pareto Analysis - top customers for 80% revenue (customer_id, full_name, customer_revenue, cumulative_revenue, cumulative_percentage)


In [26]:
q18 = """
WITH customer_revenue AS (
    SELECT
        c.customer_id,
        c.full_name,
        SUM(oi.quantity * oi.unit_price) AS customer_revenue
    FROM customers AS c
    INNER JOIN orders AS o
        ON c.customer_id = o.customer_id
    INNER JOIN order_items AS oi
        ON o.order_id = oi.order_id
    GROUP BY c.customer_id, c.full_name
),
revenue_totals AS (
    SELECT
        customer_id,
        full_name,
        customer_revenue,
        SUM(customer_revenue) OVER (
            ORDER BY customer_revenue DESC, customer_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_revenue,
        SUM(customer_revenue) OVER () AS total_revenue
    FROM customer_revenue
)
SELECT
    customer_id,
    full_name,
    ROUND(customer_revenue, 2) AS customer_revenue,
    ROUND(cumulative_revenue, 2) AS cumulative_revenue,
    ROUND((cumulative_revenue / total_revenue) * 100, 2) AS cumulative_percentage
FROM revenue_totals
WHERE (cumulative_revenue / total_revenue) <= 0.80
ORDER BY customer_revenue DESC, customer_id;
"""
q18_result = pd.read_sql(q18, engine)
q18_result

,customer_id,full_name,customer_revenue,cumulative_revenue,cumulative_percentage
0,301,Customer 301,4994.55,4994.55,0.29
1,796,Customer 796,4987.75,9982.30,0.59
2,591,Customer 591,4974.25,14956.55,0.88
3,641,Customer 641,4963.85,19920.40,1.17
4,431,Customer 431,4923.50,24843.90,1.46
...,...,...,...,...,...
508,958,Customer 958,1418.78,1357928.05,79.61
509,689,Customer 689,1410.24,1359338.29,79.69
510,771,Customer 771,1408.45,1360746.74,79.78
511,440,Customer 440,1407.84,1362154.58,79.86


### Insights
The top 513 customers account for approximately 79.94% of total revenue, which is very close to the 80% Pareto threshold. This shows that revenue is distributed across a relatively large group of high-value customers rather than only a very small group.

### Q19: Brand Affinity per Customer (customer_id, brand, brand_revenue)


In [27]:
q19 = """
SELECT
    c.customer_id,
    p.brand,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS brand_revenue
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
INNER JOIN products AS p
    ON oi.product_id = p.product_id
GROUP BY c.customer_id, p.brand
ORDER BY c.customer_id, brand_revenue DESC;
"""
q19_result = pd.read_sql(q19, engine)
q19_result

,customer_id,brand,brand_revenue
0,1,Brand 0,4760.00
1,2,Brand 1,425.43
2,3,Brand 2,448.90
3,4,Brand 3,945.48
4,5,Brand 4,3779.16
...,...,...,...
994,996,Brand 15,3342.75
995,997,Brand 16,354.93
996,998,Brand 17,1258.32
997,999,Brand 18,2727.48


### Insights
The results show the brand generating revenue for each customer. For example, Customer 1 generated 4,760.00 from Brand 0, while Customer 5 generated 3,779.16 from Brand 4, providing a basis for identifying individual brand preferences.

### Q20: Payment Failure Impact - orders with failed then successful payment (order_id, customer_id, failed_attempts, successful_payment_date)


In [28]:
q20 = """
WITH payment_summary AS (
    SELECT
        order_id,
        COUNT(CASE WHEN payment_status = 'Failed' THEN 1 END) AS failed_attempts,
        MAX(CASE WHEN payment_status = 'Completed' THEN payment_date END) AS successful_payment_date,
        MAX(CASE WHEN payment_status = 'Completed' THEN 1 ELSE 0 END) AS has_successful_payment,
        MAX(CASE WHEN payment_status = 'Completed' THEN payment_date END) AS success_date
    FROM payments
    GROUP BY order_id
)
SELECT
    o.order_id,
    o.customer_id,
    ps.failed_attempts,
    ps.successful_payment_date
FROM orders AS o
INNER JOIN payment_summary AS ps
    ON o.order_id = ps.order_id
WHERE ps.failed_attempts > 0
    AND ps.has_successful_payment = 1
ORDER BY o.order_id;
"""
q20_result = pd.read_sql(q20, engine)
q20_result

,order_id,customer_id,failed_attempts,successful_payment_date


### Insights
No orders were found with a failed payment followed by a successful payment. Therefore, the current dataset does not contain payment-retry cases matching the conditions in this query.

### Q21: Customer Order Velocity - avg days between orders (customer_id, full_name, total_orders, avg_days_between_orders)


In [29]:
q21 = """
WITH order_dates AS (
    SELECT
        o.customer_id,
        o.order_date,
        LAG(o.order_date) OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date
        ) AS previous_order_date
    FROM orders AS o
)
SELECT
    c.customer_id,
    c.full_name,
    COUNT(od.order_date) AS total_orders,
    ROUND(AVG(od.order_date - od.previous_order_date), 2) AS avg_days_between_orders
FROM customers AS c
INNER JOIN order_dates AS od
    ON c.customer_id = od.customer_id
GROUP BY c.customer_id, c.full_name
HAVING COUNT(od.order_date) > 1
ORDER BY avg_days_between_orders, c.customer_id;
"""
q21_result = pd.read_sql(q21, engine)
q21_result

,customer_id,full_name,total_orders,avg_days_between_orders


### Insights
No results were returned because the customers do not have multiple orders required to calculate the average number of days between orders. Therefore, customer order velocity cannot be calculated from the available repeat-order data.